# ☀️ Solar Seeker — YOLO26 Rooftop Segmentation Pipeline
### Complete End-to-End Training, Validation, and Solar Potential Extraction on Google Colab

This notebook guides you through training the new **YOLO26 Instance/Semantic Segmentation** model (`yolo26n-seg` / `yolo26s-seg`) from Ultralytics on aerial and satellite rooftop imagery.

#### 📋 Workflow Overview:
1. **Hardware & Environment**: Verify GPU availability and install `ultralytics`.
2. **Dataset Setup**: Unpack and automatically validate the YOLO polygon segmentation dataset.
3. **Pre-Training Sanity Check**: Overlay ground-truth rooftop polygons on satellite imagery.
4. **YOLO26 Model Training**: Train `yolo26n-seg` with aerial-optimized augmentations.
5. **Performance Evaluation**: Review Mask mAP50, mAP50-95, and loss curves.
6. **Visual Inference**: Run inference on unseen satellite images and render predictions.
7. **Solar Seeker Physics Engine**: Extract usable area ($m^2$), peak capacity ($kWp$), and annual yield.
8. **Export & Download**: Package `best.pt`, ONNX model, and training plots for local deployment.


## 1. Setup Environment & Verify GPU Hardware
Make sure your Colab runtime has a GPU enabled:
> **Runtime** ➔ **Change runtime type** ➔ **Hardware accelerator** ➔ **T4 GPU** (or A100).


In [ ]:
# 1. Check GPU allocation
!nvidia-smi

# 2. Fix Pillow / PIL compatibility & install Ultralytics
# Colab pre-loads an older Pillow in memory; force-reinstalling ensures consistent modules.
!pip install -q --upgrade --force-reinstall "pillow>=10.4.0,<12.0.0"
!pip install -q --upgrade ultralytics opencv-python-headless matplotlib pyyaml

# 3. Verify Pillow import integrity
try:
    import PIL
    from PIL import Image, ImageDraw, ImageFont
    import ultralytics
    from ultralytics import YOLO
    import torch
    print(f"✅ PyTorch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"✅ Pillow version: {PIL.__version__}")
    print(f"✅ Ultralytics version: {ultralytics.__version__}")
except ImportError as e:
    print(f"⚠️ In-memory Pillow mismatch detected: {e}")
    print("🔄 Automatically restarting Colab runtime to reload clean modules... (wait 3 seconds and re-run)")
    import os
    os.kill(os.getpid(), 9)


## 2. Dataset Ingestion & Validation
Upload your dataset archive (`solar_seeker_yolo_dataset.zip` or `solar-seeker-anno-yolo.zip`).
You can either:
- **Option A:** Upload directly using the file browser on the left panel (📁 icon).
- **Option B:** Run the cell below to prompt an upload widget.
- **Option C:** Mount Google Drive if your dataset is stored there.


In [ ]:
import os
import sys
import shutil
import zipfile
import glob
from pathlib import Path
import yaml

DATASET_ROOT = Path("/content/dataset")

# Check for zip files in current directory
zip_candidates = [
    "solar_seeker_yolo_dataset.zip",
    "solar-seeker-anno-yolo.zip",
    "solar-seeker-anno.zip",
    "dataset.zip"
]

zip_file = None
for z in zip_candidates:
    if os.path.exists(z):
        zip_file = z
        break

if not zip_file and not DATASET_ROOT.exists():
    try:
        from google.colab import files
        print("📥 Please upload your dataset zip file (e.g., 'solar_seeker_yolo_dataset.zip'):")
        uploaded = files.upload()
        for f in uploaded.keys():
            if f.endswith(".zip"):
                zip_file = f
                break
    except Exception as e:
        print(f"Colab upload prompt skipped: {e}")

if zip_file and os.path.exists(zip_file):
    print(f"📦 Unzipping '{zip_file}' to '{DATASET_ROOT}'...")
    if DATASET_ROOT.exists():
        shutil.rmtree(DATASET_ROOT)
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_file, "r") as z:
        z.extractall(DATASET_ROOT)
    print("✅ Extraction complete!")

# Dataset Verification & Auto-Structuring
train_img_dir = DATASET_ROOT / "images" / "train"
train_lbl_dir = DATASET_ROOT / "labels" / "train"
val_img_dir = DATASET_ROOT / "images" / "val"
val_lbl_dir = DATASET_ROOT / "labels" / "val"

# If dataset was extracted without images/train subdirs, search recursively
if not train_img_dir.exists():
    print("🔍 Auto-detecting image and label directory structure...")
    valid_exts = {".jpg", ".jpeg", ".png", ".webp"}
    all_imgs = [p for p in DATASET_ROOT.rglob("*") if p.suffix.lower() in valid_exts]
    all_txts = [p for p in DATASET_ROOT.rglob("*.txt") if p.name not in ("train.txt", "val.txt", "classes.txt")]

    if all_imgs and all_txts:
        print(f"Found {len(all_imgs)} images and {len(all_txts)} label files. Structuring standard YOLO directories...")
        train_img_dir.mkdir(parents=True, exist_ok=True)
        train_lbl_dir.mkdir(parents=True, exist_ok=True)
        val_img_dir.mkdir(parents=True, exist_ok=True)
        val_lbl_dir.mkdir(parents=True, exist_ok=True)

        matched_pairs = []
        for img in all_imgs:
            matching_lbl = next((l for l in all_txts if l.stem == img.stem), None)
            if matching_lbl:
                matched_pairs.append((img, matching_lbl))

        print(f"Matched {len(matched_pairs)} image-label pairs.")
        # Split 80% train, 20% val (8 train, 2 val)
        val_count = max(1, int(len(matched_pairs) * 0.2)) if len(matched_pairs) > 3 else 1
        train_pairs = matched_pairs[:-val_count]
        val_pairs = matched_pairs[-val_count:]

        for img, lbl in train_pairs:
            shutil.copy2(img, train_img_dir / img.name)
            shutil.copy2(lbl, train_lbl_dir / lbl.name)

        for img, lbl in val_pairs:
            shutil.copy2(img, val_img_dir / img.name)
            shutil.copy2(lbl, val_lbl_dir / lbl.name)

# Ensure val split has at least 1 image to prevent YOLO validation error
train_images = list(train_img_dir.glob("*.*")) if train_img_dir.exists() else []
val_images = list(val_img_dir.glob("*.*")) if val_img_dir.exists() else []

if train_images and not val_images:
    val_img_dir.mkdir(parents=True, exist_ok=True)
    val_lbl_dir.mkdir(parents=True, exist_ok=True)
    for img in train_images[-2:]:
        shutil.copy2(img, val_img_dir / img.name)
        lbl = train_lbl_dir / f"{img.stem}.txt"
        if lbl.exists():
            shutil.copy2(lbl, val_lbl_dir / lbl.name)
    val_images = list(val_img_dir.glob("*.*"))

# Generate / Update data.yaml
data_yaml_path = DATASET_ROOT / "data.yaml"
data_config = {
    "path": str(DATASET_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "roof"}
}

with open(data_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f"✅ data.yaml configured at: {data_yaml_path}")
print(f"📊 Dataset Summary:")
print(f"   • Train images: {len(train_images)} ({train_img_dir})")
print(f"   • Val images:   {len(val_images)} ({val_img_dir})")
print(f"   • Classes:      {data_config['names']}")


## 3. Ground-Truth Annotation Sanity Check
Before training, let's visualize the rooftop polygon annotations drawn on top of the satellite imagery to verify coordinate alignment.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def visualize_annotations(img_dir, lbl_dir, num_samples=3):
    img_paths = sorted(list(img_dir.glob("*.*")))[:num_samples]
    if not img_paths:
        print("No images found to visualize.")
        return

    fig, axes = plt.subplots(1, len(img_paths), figsize=(6 * len(img_paths), 6))
    if len(img_paths) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, img_paths):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        overlay = img.copy()

        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        poly_count = 0
        if lbl_path.exists():
            with open(lbl_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cls_id = int(parts[0])
                    coords = [float(x) for x in parts[1:]]
                    pts = np.array(coords).reshape(-1, 2)
                    pts[:, 0] = pts[:, 0] * w
                    pts[:, 1] = pts[:, 1] * h
                    pts = pts.astype(np.int32)

                    cv2.fillPoly(overlay, [pts], (255, 140, 0))
                    cv2.polylines(img, [pts], isClosed=True, color=(0, 255, 255), thickness=2)
                    poly_count += 1

        blended = cv2.addWeighted(overlay, 0.4, img, 0.6, 0)
        ax.imshow(blended)
        ax.set_title(f"{img_path.name[:25]}... ({poly_count} roofs)", fontsize=11)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

visualize_annotations(train_img_dir, train_lbl_dir, num_samples=3)


## 4. Train YOLO26 Segmentation Model
We initialize `yolo26n-seg.pt` (Nano) or `yolo26s-seg.pt` (Small) pre-trained on COCO segmentation.
Key hyperparameters configured for rooftop satellite imagery:
- **`imgsz=640`**: Balanced input resolution for high-altitude rooftop details.
- **`epochs=60`**: Ample convergence time for fine-tuning.
- **`batch=4`**: Memory-efficient for Colab T4 GPU.
- **`degrees=15.0`, `flipud=0.5`, `fliplr=0.5`**: Simulates diverse aerial flyover angles.
- **`mosaic=1.0`**: Enhances multi-scale rooftop detection.


In [ ]:
from ultralytics import YOLO

MODEL_WEIGHTS = "yolo26n-seg.pt"

try:
    print(f"🚀 Initializing pre-trained model: {MODEL_WEIGHTS}...")
    model = YOLO(MODEL_WEIGHTS)
except Exception as e:
    print(f"⚠️ Notice: {MODEL_WEIGHTS} fallback triggered ({e}).")
    print("Falling back to yolo11n-seg.pt...")
    model = YOLO("yolo11n-seg.pt")

# Launch training
train_results = model.train(
    data=str(data_yaml_path),
    task="segment",
    epochs=60,
    imgsz=640,
    batch=4,
    device=0,             # GPU 0
    workers=2,
    patience=25,
    optimizer="AdamW",
    lr0=0.001,
    augment=True,
    degrees=15.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    save=True,
    plots=True,
    name="solar_seeker_yolo26_run"
)

print("🎉 Training finished successfully!")


## 5. Model Evaluation & Validation Curves
Examine quantitative segmentation metrics: Mask mAP50, Mask mAP50-95, Box mAP, Precision, and Recall.


In [ ]:
import glob
from IPython.display import Image, display

# Evaluate on validation set
metrics = model.val()

print("\n" + "="*50)
print("📊 FINAL VALIDATION METRICS:")
print("="*50)
print(f"  • Mask mAP@50:      {metrics.seg.map50:.4f}")
print(f"  • Mask mAP@50-95:   {metrics.seg.map:.4f}")
print(f"  • Box mAP@50:       {metrics.box.map50:.4f}")
print(f"  • Box mAP@50-95:    {metrics.box.map:.4f}")
print("="*50)

# Display Ultralytics training progression charts (results.png)
results_png_list = glob.glob("runs/segment/solar_seeker_yolo26_run*/results.png")
if results_png_list:
    print("📈 Training Loss and mAP Progression Curves:")
    display(Image(filename=results_png_list[-1]))


## 6. Real Satellite Image Inference
Run predictions on validation satellite images and visualize the detected rooftop footprints.


In [ ]:
# Run inference using the best saved weights
best_weights = glob.glob("runs/segment/solar_seeker_yolo26_run*/weights/best.pt")[-1]
inference_model = YOLO(best_weights)

# Predict on validation images
val_sample_imgs = list(val_img_dir.glob("*.*"))[:3]
predictions = inference_model.predict(
    source=[str(p) for p in val_sample_imgs],
    conf=0.25,
    iou=0.5,
    save=False
)

fig, axes = plt.subplots(len(predictions), 2, figsize=(14, 5 * len(predictions)))
if len(predictions) == 1:
    axes = np.expand_dims(axes, 0)

for i, (pred, img_p) in enumerate(zip(predictions, val_sample_imgs)):
    orig_bgr = cv2.imread(str(img_p))
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)
    
    # Plot Original
    axes[i, 0].imshow(orig_rgb)
    axes[i, 0].set_title(f"Original: {img_p.name}", fontsize=11)
    axes[i, 0].axis("off")
    
    # Plot YOLO26 Prediction
    pred_rgb = pred.plot(boxes=True, masks=True)
    pred_rgb = cv2.cvtColor(pred_rgb, cv2.COLOR_BGR2RGB)
    
    count = len(pred.masks) if pred.masks is not None else 0
    axes[i, 1].imshow(pred_rgb)
    axes[i, 1].set_title(f"YOLO26 Prediction ({count} roofs detected)", fontsize=11)
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


## 7. Solar Seeker Physics & Solar Energy Engine
Convert YOLO26 segmentation masks into real-world solar metrics:
- **Ground Sample Distance (GSD)**: $0.15\text{ m/px}$ (at Zoom Level 19).
- **Rooftop Utilization Factor ($k$)**: $65\%$ ($0.65$) for edge setbacks and maintenance paths.
- **Panel Density**: $0.20\text{ kWp/m}^2$ (standard commercial monocrystalline panels).
- **Estimated Annual Yield**: $\approx 1,350\text{ kWh/kWp}$ (Istanbul / Mediterranean climate).


In [ ]:
GSD = 0.15                   # Ground Sample Distance: meters per pixel
UTILIZATION_FACTOR = 0.65    # 65% usable solar area
PANEL_DENSITY_KWP = 0.20     # 0.20 kWp per m2
ANNUAL_YIELD_KWH_PER_KWP = 1350.0 # kWh per kWp installed

print(f"{'Roof #':<8} | {'Gross Area (m²)':<16} | {'Net Usable (m²)':<16} | {'Capacity (kWp)':<16} | {'Annual Yield (kWh)':<18}")
print("-" * 82)

total_gross_m2 = 0.0
total_net_m2 = 0.0
total_kwp = 0.0

if predictions and predictions[0].masks is not None:
    polygons = predictions[0].masks.xy
    
    for idx, poly in enumerate(polygons, 1):
        if len(poly) < 3:
            continue
        pixel_area = cv2.contourArea(poly.astype(np.float32))
        gross_area_m2 = pixel_area * (GSD ** 2)
        net_area_m2 = gross_area_m2 * UTILIZATION_FACTOR
        kwp = net_area_m2 * PANEL_DENSITY_KWP
        annual_kwh = kwp * ANNUAL_YIELD_KWH_PER_KWP
        
        total_gross_m2 += gross_area_m2
        total_net_m2 += net_area_m2
        total_kwp += kwp
        
        print(f"{idx:<8} | {gross_area_m2:<16.2f} | {net_area_m2:<16.2f} | {kwp:<16.2f} | {annual_kwh:<18.1f}")

print("=" * 82)
print(f"TOTALS   | {total_gross_m2:<16.2f} | {total_net_m2:<16.2f} | {total_kwp:<16.2f} | {(total_kwp * ANNUAL_YIELD_KWH_PER_KWP):<18.1f}")


## 8. Export Weights & Download Training Package
Export the best weights to **ONNX** format for high-speed inference, and bundle all checkpoints and training plots into `solar_seeker_yolo26_trained.zip` for instant download.


In [ ]:
import zipfile
from google.colab import files

best_weights_path = glob.glob("runs/segment/solar_seeker_yolo26_run*/weights/best.pt")[-1]
export_model = YOLO(best_weights_path)

# Export to ONNX
print("⚡ Exporting model to ONNX format...")
try:
    onnx_path = export_model.export(format="onnx", imgsz=640)
except Exception as e:
    print(f"ONNX export notice: {e}")
    onnx_path = None

# Package into zip archive
output_archive = "solar_seeker_yolo26_trained.zip"
print(f"📦 Packaging artifacts into {output_archive}...")

with zipfile.ZipFile(output_archive, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(best_weights_path, arcname="best.pt")
    if onnx_path and os.path.exists(str(onnx_path)):
        z.write(str(onnx_path), arcname="best.onnx")
    if data_yaml_path.exists():
        z.write(str(data_yaml_path), arcname="data.yaml")
    for plot_file in glob.glob("runs/segment/solar_seeker_yolo26_run*/*.png"):
        z.write(plot_file, arcname=os.path.basename(plot_file))

print(f"✅ Archive created ({os.path.getsize(output_archive) / 1024 / 1024:.2f} MB).")
print("⬇️ Triggering browser download...")
try:
    files.download(output_archive)
except Exception as e:
    print(f"Download trigger notice: {e}")
    print(f"You can manually right-click '{output_archive}' in the left Colab files panel to download.")
